# Phase 1: MVP, a coding agent that can fix a bug safely

Phase 0's agent can only **read**. It can explain a bug but not fix it. This phase gives it the tools to explore, edit and run code, with safety checks before anything risky.

This notebook is the lab: each step's plan, evidence, code and observations live here. [phase_1_README.md](phase_1_README.md) is written once the phase is finished (see [WORKFLOW.md](../WORKFLOW.md)).

## Goal

**Done when:** given a small project with a failing test, the agent

1. finds the bug,
2. fixes it,
3. runs the tests to confirm the fix,
4. and asks before anything risky.

## Plan

One section per roadmap step, built strictly in this order. Each step starts from what already exists and changes only what it needs.

| # | Section | What it adds | Why at this point |
|---|---|---|---|
| 1 | Tool registry | One place that lists the tools; the prompt, parser and loop read from it | Set up before adding tools, so each new tool is a function, not a prompt rewrite |
| 2 | List and find files | Tools to see what's in a project | The agent has to explore a project before it can change it |
| 3 | Folder boundary | File tools refuse paths outside the project | Before any tool can write |
| 4 | Write and edit files | Tools that change files | The moment the agent can actually change code |
| 5 | Approval prompt | Ask the user before risky actions | Needed before shell access |
| 6 | Run shell commands | A tool that runs commands | Lets the agent run code, tests and git |
| 7 | Verify loop | Run the tests, fix what fails, repeat | What makes it a *coding* agent |

After section 7: an end-to-end demo against the goal above. Then this notebook's code is refactored into `agent_v1/` (see [WORKFLOW.md](../WORKFLOW.md)).

## How this notebook works

- It imports **only** Phase 0's frozen package, `agent_v0`. Nothing comes from `agent_v1`, which doesn't exist yet.
- Everything new is built here. To change a Phase 0 function, we look at the imported version, then write a new one **with the same name** in this notebook (*shadowing*). We never modify `agent_v0` itself.
- **Every step starts with a game plan**: the problem, how real agents solve it, the proposed sub-steps, decisions and acceptance criteria. We agree on it before any code, then build the sub-steps one at a time.

## 0. Setup: import Phase 0

`phase-0-foundation` has a hyphen in its name, so Python can't import it as a package. Instead we add the folder to Python's search path, then import `agent_v0` from inside it.

In [ ]:
import sys

sys.path.insert(0, "../phase-0-foundation")

from agent_v0 import SYSTEM_PROMPT, ask, call_llm, new_conversation, parse_tool_call, read_file

A quick check that it's the Phase 0 agent we know: its prompt still describes one hand-written tool.

In [ ]:
print(SYSTEM_PROMPT)

## 1. Tool registry: game plan

*Draft for review. Nothing below is built yet; we agree on this plan first.*

### What we're doing and why

Phase 0 has one tool, `read_file`, and its name is written by hand in **three places** in `agent_v0/loop.py`:

| Place | Lines | What's hard-coded |
|---|---|---|
| The system prompt | [`loop.py:10-14`](../phase-0-foundation/agent_v0/loop.py) | "You have ONE tool: read_file(filename) ..." |
| The parser | [`loop.py:22`](../phase-0-foundation/agent_v0/loop.py) | `prefix = "tool: read_file("` |
| The loop | [`loop.py:51-52`](../phase-0-foundation/agent_v0/loop.py) | `result = read_file(args.get("filename", ""))` |

So adding a second tool would mean editing all three. The roadmap's goal for this step ([`ROADMAP.md:181`](../ROADMAP.md)): *"Describe tools automatically from their code; add a tool without touching the loop."* It comes first in Phase 1 because steps 2, 4 and 6 all add tools ([`ROADMAP.md:70`](../ROADMAP.md)).

### Evidence: how real agents do it

The notebook is how *we* do it. This table is how everyone else does: small-scale projects first, then the five big open-source industry agents (Gemini CLI, Codex CLI, opencode, OpenHands, Aider). Source code read on 2026-10-31; the opencode, OpenHands and Aider links are pinned to the commit read.

| Tier | Agent | How it registers tools | Source |
|---|---|---|---|
| Small scale | **learn-claude-code** (lesson s02) | Two separate pieces: a hand-written list of tool descriptions (`TOOLS`) and a dictionary from name to function (`TOOL_HANDLERS`). The loop calls `TOOL_HANDLERS[name](**args)`; an unknown name returns `"Unknown: <name>"`. Its own summary: *"the loop stays the same; only tool registration and dispatch grow."* | [`s02_tool_use/code.py:138-156, 180-181`](https://github.com/shareAI-lab/learn-claude-code/blob/HEAD/s02_tool_use/code.py#L138-L181) |
| Small scale | **ghuntley/how-to-build-a-coding-agent** (Go) | One bundle per tool, `ToolDefinition`: name, description, input schema and function together. The description is hand-written; the input schema is **generated** from a type. The loop searches the tool list by name; an unknown name returns "tool not found". | [`read.go:253-271, 147-170`](https://github.com/ghuntley/how-to-build-a-coding-agent/blob/HEAD/read.go#L253-L271) |
| Small scale | **mini-swe-agent** (text mode) | **No registry.** One action type, a shell command, pulled out of the reply with a regex. A badly formatted reply is sent back to the model as an error message. | [`actions_text.py:15-40`](https://github.com/SWE-agent/mini-swe-agent/blob/HEAD/src/minisweagent/models/utils/actions_text.py#L15-L40) |
| Industry | **Gemini CLI** (Google, TypeScript, ~109k stars) | A `ToolRegistry` class holding a map from name to tool object; `registerTool` overwrites a duplicate name with a warning. Tools come from three sources: built in, **discovered** by running a project command, and MCP servers. The list sent to the model is rebuilt from each tool's own schema, **minus tools excluded in config**; in Plan Mode it **rewrites the write and edit tools' descriptions** to "only for plans". An unknown tool returns `Tool "x" not found.` plus **"did you mean" suggestions**: the 3 closest names by edit distance. Descriptions are hand-written in separate files per model family, and **the same tool gets different descriptions for different models**: for `read_file`, the Gemini 3 set says the model "MUST use 'start_line' and 'end_line' for targeted, surgical reads"; the default set doesn't. | [`tool-registry.ts:247-294`](https://github.com/google-gemini/gemini-cli/blob/main/packages/core/src/tools/tool-registry.ts#L247-L294), [`:370-530`](https://github.com/google-gemini/gemini-cli/blob/main/packages/core/src/tools/tool-registry.ts#L370-L530), [`:663-718`](https://github.com/google-gemini/gemini-cli/blob/main/packages/core/src/tools/tool-registry.ts#L663-L718); [`scheduler.ts:374-419`](https://github.com/google-gemini/gemini-cli/blob/main/packages/core/src/scheduler/scheduler.ts#L374-L419); [`tool-utils.ts:65`](https://github.com/google-gemini/gemini-cli/blob/main/packages/core/src/utils/tool-utils.ts#L65); [`default-legacy.ts:84-86`](https://github.com/google-gemini/gemini-cli/blob/main/packages/core/src/tools/definitions/model-family-sets/default-legacy.ts#L84-L86), [`gemini-3.ts:93-95`](https://github.com/google-gemini/gemini-cli/blob/main/packages/core/src/tools/definitions/model-family-sets/gemini-3.ts#L93-L95), [`resolver.ts:17-34`](https://github.com/google-gemini/gemini-cli/blob/main/packages/core/src/tools/definitions/resolver.ts#L17-L34) |
| Industry | **Codex CLI** (OpenAI, Rust, ~132k stars) | A `ToolRegistry` holding an ordered map from name to tool. **Two ways in**: `register_trusted` for built-in tools and `register_external` for outside tools, which must first pass a **tool policy**. Each built-in tool's description and parameters are **hand-written in code**, and vary by situation (the shell tool adds Windows guidance; approval parameters are added to its schema). An unknown tool returns `"unsupported call: <name>"` to the model, and the failed call is **logged to telemetry** (OpenTelemetry). | [`registry.rs:428-433, 516-580`](https://github.com/openai/codex/blob/main/codex-rs/core/src/tools/registry.rs#L428-L580), [`:766-786, 1148-1151`](https://github.com/openai/codex/blob/main/codex-rs/core/src/tools/registry.rs#L766-L786); [`shell_spec.rs:14-108`](https://github.com/openai/codex/blob/main/codex-rs/core/src/tools/handlers/shell_spec.rs#L14-L108) |
| Industry | **opencode** (independent, TypeScript, ~224k stars) | A registry keyed by tool name, with an editor API to add, update and remove tools; each tool is **validated when it's registered** (bad names or schemas become registration errors). Before each model call it takes a **snapshot** of the tools, dropping any that the agent's **permission rules** deny, so the model never sees them. Each parameter carries its own hand-written description. An unknown tool returns *"No tool named "x" is currently available. Please use a tool from the available tool list."* It also has a **Code Mode**: instead of calling tools one at a time, the model writes a small program that calls them. | [`tool.ts:22-42`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L22-L42), [`:265-290`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L265-L290), [`:328-335`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L328-L335), [`:381-414`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L381-L414); [`tool/plugin/read.ts:17-43`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool/plugin/read.ts#L17-L43) |
| Industry | **OpenHands** (All Hands AI, Python SDK; main repo ~92k stars) | A **Python dictionary** from name to factory: `register_tool(name, factory)` takes a ready tool or a class that builds tools per conversation; a duplicate name logs a warning and overwrites. Each tool's arguments are a typed (Pydantic) class whose fields carry hand-written descriptions, and the terminal tool has **separate Unix and Windows descriptions**. Ships **model-specific tool sets** (e.g. Gemini-style file tools). An unknown tool returns *"Tool 'x' not found. Available: ..."*; **wrong arguments return a validation error to the model**, and it even tries to detect and repair some malformed calls. | [`tool/registry.py:34, 130-175`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-sdk/openhands/sdk/tool/registry.py#L130-L175); [`agent/agent.py:1414-1417, 1481`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-sdk/openhands/sdk/agent/agent.py#L1414-L1481); [`terminal/definition.py:56-102`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-tools/openhands/tools/terminal/definition.py#L56-L102); [`tools/gemini/`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-tools/openhands/tools/gemini) |
| Industry | **Aider** (Python, ~50k stars) | **No tool registry, by design.** The model doesn't call tools to edit; it writes edits in a **text format** (whole files, search/replace blocks, unified diffs), which Aider parses and applies. The "registry" is a list of these *edit formats*, picked by name; an unknown one is an error listing the valid formats. Function-calling versions exist in the code but are **commented out** of the active list. | [`coders/__init__.py:16-34`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/__init__.py#L16-L34); [`base_coder.py:61, 190-191`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/base_coder.py#L190-L191); [`editblock_coder.py:18-41, 439`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/editblock_coder.py#L18-L41); [`wholefile_func_coder.py:8-25`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/wholefile_func_coder.py#L8-L25) |

**What the six agents with a tool registry agree on, and so what this plan follows:**
- **A lookup from tool name to tool**, and the loop finds tools by name (1b, 1e). Every agent with a registry does this, from 150-line tutorials to Gemini CLI, Codex, opencode and OpenHands.
- **An unknown tool is an error the model sees**, not a crash (1e). The better ones tell the model what *is* available (OpenHands lists the tools, opencode points to the tool list, Gemini CLI suggests the closest names). This also matches Phase 0's own rule: `read_file` returns errors instead of raising them.

**Where they differ:**
- **Where descriptions come from.** All six agents with a registry hand-write their descriptions. Only ghuntley and OpenHands generate anything automatically, and only the argument schema, from a typed class. Two industry agents go further the other way: Gemini CLI and OpenHands keep **different descriptions or tool sets for different models**, treating descriptions as prompt text to tune, not just documentation. This bears on decision 1 below, and on Phase 2's "same harness, different models" ([`ROADMAP.md:90`](../ROADMAP.md)).
- **Industry tool lists aren't fixed.** In Gemini CLI, Codex and opencode, which tools the model sees depends on config, policy, permissions or mode, and tools can arrive at runtime (discovery, MCP, external tools). The small projects have a fixed list. We'll meet the same pressures later: the approval prompt is step 5, MCP is step 14 ([`ROADMAP.md:100`](../ROADMAP.md)).
- **Bad arguments.** OpenHands validates arguments against the tool's schema and returns the error to the model, and tries to repair some malformed calls. This bears on decision 3 below.
- **Unknown-tool errors get smarter at scale.** The small projects just say "unknown"; the industry agents list or suggest alternatives, and Codex records the failure in telemetry, which is the kind of record Phase 2 (measurement) builds.
- **Text protocol vs native tool calling.** All the registry agents use **native tool calling** (the API receives the tool list and returns structured requests). We keep our text protocol for now; native tool calling is step 12, and comparing the two is step 13 ([`ROADMAP.md:98-99`](../ROADMAP.md)).
- **No registry at all** is a real design too, in two very different ways: mini-swe-agent has one shell tool (the roadmap's "Bash-only agent", Later, [`ROADMAP.md:187`](../ROADMAP.md)), and **Aider**, a major production agent, edits through text formats instead of tool calls, with its function-calling coders commented out. Text protocols like ours aren't just a teaching shortcut.

Not covered: Claude Code and Cursor aren't open source, so they'd appear only through their public docs; not read for this step.

### Sub-steps (proposed)

Proposed from the problem and the evidence above. Each one is a separate turn: built, run and discussed before the next.

| # | Sub-step | Replaces | Based on |
|---|---|---|---|
| 1a | **Look at the problem.** Print Phase 0's parser and `ask()` (its prompt is already printed in the setup above), and find the three hard-coded places. | — | The problem table above |
| 1b | **The registry.** One place that lists the tools: a dictionary from tool name to function. Register Phase 0's `read_file`, unchanged. | — | All six agents with a registry keep a lookup from name to tool (mini-swe-agent and Aider have none) |
| 1c | **Describe tools from the registry.** Build the prompt's tool list from what's registered, instead of typing it. | `loop.py:10-14` | Gemini CLI builds the model's tool list from the registry; *how* each description is written is decision 1 |
| 1d | **Parse any tool name.** The parser accepts `tool: <name>({...})` and returns which tool was asked for, plus its arguments. | `loop.py:22` | Our text protocol (the native-calling agents get this from the API); mini-swe-agent and Aider also parse text |
| 1e | **Run a tool by name.** Look it up in the registry and call it. An unknown tool or wrong arguments come back as an error result, not a crash. Try both failure cases: an unknown tool name, and wrong arguments. | `loop.py:51-52` | All six agents with a registry return unknown tools as errors the model sees; OpenHands also returns argument errors; Phase 0's errors-as-results rule; "done when" check 3 |
| 1f | **The new loop.** Phase 0's `ask()`, rewritten in this notebook to use 1c–1e. | `ask()` | learn-claude-code: *"the loop stays the same; only tool registration and dispatch grow"* |
| 1g | **Try it.** Phase 0's demo question, then print the conversation. Then register a tiny throwaway tool and check that it appears in the prompt and can be called, with no other change. | — | "Done when" checks 1 and 2 |

Not proposed for this step, though industry agents do them: tools filtered by config or mode, tools added at runtime, "did you mean" suggestions, telemetry. These belong to later steps (approval, MCP, measurement), or could be added if you want them now.

### Decisions for you

These are design choices, not settled by the evidence. Each is my judgment until you decide.

1. **Where does a tool's description come from?** (affects 1b, 1c)
   - *Hand-written, next to the registry*, like learn-claude-code: full control over the wording, but code and description can drift apart.
   - *Generated from the function itself*, from its name, parameter names and docstring: closest to the roadmap's "describe tools automatically from their code", and to ghuntley generating the schema. The description can only be as good as the docstring.
   - Evidence: **all six agents with a registry hand-write descriptions**, including all four with a registry among the big five, and Gemini CLI and OpenHands tune them per model. The roadmap's "automatically" (`ROADMAP.md:181`) goes against what everyone actually does, which is worth deciding consciously.
2. **Should the prompt show a concrete example line** (as Phase 0 does, `loop.py:14`) **or only a generic pattern?** (affects 1c) A concrete example is probably followed more reliably. In the code I wrote too early today (since removed), a generic pattern let the model send a bare string instead of `{"filename": ...}` in 1 of 4 test runs. That's one small observation, not a measurement.
3. **What if the arguments aren't a JSON object?** (affects 1d, 1e) Reject them in the parser with a clear error, like mini-swe-agent's format errors, or let 1e's "bad arguments" error catch it. OpenHands does the most here: it validates arguments against each tool's schema and returns the error to the model.

### Done when

- With only `read_file` registered, the agent answers Phase 0's question the same way as before.
- Adding a tool needs only registering a function: no change to the prompt, parser or loop.
- Asking for an unknown tool, or passing wrong arguments, gives the model an error result instead of crashing.
